# Directed Biological Analyses of Vitamin D-Related Signatures

This notebook presents the **core biological analyses** of transcriptomic responses to vitamin D and its analogs, based on the LINCS L1000 dataset.  
While the previous notebooks focused on data preparation, quality control, and exploratory analysis, here we shift to **hypothesis-driven investigations**.

**Objectives of this notebook:**
- Quantify dose–response effects at the gene level.  
- Identify eligible *high vs. low dose* contrasts across compounds and cell lines.  
- Perform gene set and pathway enrichment analyses.  
- Compare analogs to distinguish **shared effects** from **compound-specific responses**.  

In [ ]:
# Core scientific stack
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Statistics and modeling
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova
from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities (for scaling or decomposition if needed)
from sklearn.preprocessing import StandardScaler

# Configure plotting aesthetics
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")
%matplotlib inline

## Data Setup

We begin by loading all project data tables into memory:  
- **Expression matrix** (genes × signatures).  
- **Signature metadata** (perturbation, dose, cell line, etc.).  
- **Compound metadata** (compound-level annotations).  
- **Cell line metadata** (cell type, lineage, disease).  
- **Gene metadata** (landmark vs. inferred, gene symbols).  

Having all tables available ensures that downstream analyses can seamlessly combine expression values with their biological and experimental context.


In [ ]:
# Define data directory and file paths
DATA_DIR = "../data/exports"

PATHS = {
    "exp":   f"{DATA_DIR}/expression_matrix_clean.parquet",   # expression matrix
    "sig":   f"{DATA_DIR}/signature_metadata_clean.csv",      # signature metadata
    "comp":  f"{DATA_DIR}/subset_compounds_meta.csv",         # compounds
    "cells": f"{DATA_DIR}/subset_cell_lines_meta.csv",        # cell lines
    "genes": f"{DATA_DIR}/subset_genes_meta.csv",             # genes
}

# Load all tables into memory
exp_matrix = pd.read_parquet(PATHS["exp"])
metadata   = pd.read_csv(PATHS["sig"])
compounds  = pd.read_csv(PATHS["comp"])
cell_lines = pd.read_csv(PATHS["cells"])
gene_info  = pd.read_csv(PATHS["genes"])

# Quick overview of dimensions
print(f"Expression matrix: {exp_matrix.shape[0]} genes × {exp_matrix.shape[1]} signatures")
print(f"Metadata rows:     {len(metadata)}")
print(f"Compounds:         {len(compounds)}")
print(f"Cell lines:        {len(cell_lines)}")
print(f"Genes:             {len(gene_info)}")


### Data Setup Conclusion

All data tables were successfully loaded:  
- Expression matrix with 12,328 genes × 258 signatures  
- 258 metadata entries  
- 12 compounds, 5 cell lines, and 12,328 genes  

The dataset is ready for downstream analyses.

---

## Integrity Gatekeeper

Before performing directed analyses, we run a minimal integrity check to ensure that the expression matrix and metadata are fully aligned and free of basic issues.  
This step verifies:  
- Consistent signature identifiers across tables  
- No missing values or zero-variance features  
- Dose information available and usable  


In [ ]:
def minimal_gatekeeper(exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=None):
    # Identify signature ID column in metadata
    sig_id_col = next((c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata.columns), None)
    if sig_id_col is None:
        raise ValueError("Signature ID column not found in metadata.")
    
    # Expression–metadata alignment (same set and order of signatures)
    exp_cols = pd.Index(map(str, exp_matrix.columns))
    meta_ids = pd.Index(metadata[sig_id_col].astype(str))
    common = exp_cols.intersection(meta_ids)
    if expected_n is not None and len(common) != expected_n:
        raise AssertionError(f"Common signatures = {len(common)} (expected {expected_n}).")
    if len(exp_cols.difference(common)) or len(meta_ids.difference(common)):
        raise AssertionError("Expression and metadata do not contain the exact same signatures.")
    meta_aligned = metadata.set_index(sig_id_col).loc[exp_cols].reset_index().rename(columns={"index": sig_id_col})
    
    # Basic integrity: NA and zero variance
    if exp_matrix.isna().any().any():
        raise AssertionError("NA values found in expression matrix.")
    if (exp_matrix.var(axis=1) == 0).any():
        raise AssertionError("Zero-variance genes detected.")
    if (exp_matrix.var(axis=0) == 0).any():
        raise AssertionError("Zero-variance signatures detected.")
    
    # Dose usability: numeric and variable within groups
    dose_col = next((c for c in meta_aligned.columns if ("dose" in c.lower()) and ("unit" not in c.lower())), None)
    if dose_col is None:
        raise AssertionError("Numeric dose column not found in metadata.")
    meta_aligned["dose_value"] = pd.to_numeric(meta_aligned[dose_col], errors="coerce")
    if meta_aligned["dose_value"].isna().any():
        raise AssertionError("Non-numeric values in dose column.")
    group_keys = [k for k in ["pert_id", "cell_id"] if k in meta_aligned.columns]
    if not group_keys:
        raise AssertionError("Missing grouping keys (pert_id/cell_id).")
    var_by_group = meta_aligned.groupby(group_keys)["dose_value"].agg(lambda x: float(np.var(x, ddof=1)) if x.notna().any() else 0.0)
    if (var_by_group == 0).all():
        raise AssertionError("No within-group dose variation; dose–response analyses are not feasible.")
    
    # Referential checks against lookup tables (lightweight)
    if "pert_id" in meta_aligned.columns and "pert_id" in compounds.columns:
        missing_comp = set(meta_aligned["pert_id"]) - set(compounds["pert_id"])
        if missing_comp:
            raise AssertionError(f"Missing compound keys in 'compounds': {len(missing_comp)}.")
    if "cell_id" in meta_aligned.columns and "cell_id" in cell_lines.columns:
        missing_cells = set(meta_aligned["cell_id"]) - set(cell_lines["cell_id"])
        if missing_cells:
            raise AssertionError(f"Missing cell IDs in 'cell_lines': {len(missing_cells)}.")
    if "gene_id" in getattr(gene_info, "columns", []):
        missing_genes = set(map(str, exp_matrix.index)) - set(map(str, gene_info["gene_id"]))
        if missing_genes:
            raise AssertionError(f"Missing gene IDs in 'gene_info': {len(missing_genes)}.")
    
    summary = {
        "signatures": len(common),
        "genes": exp_matrix.shape[0],
        "dose_col": dose_col,
        "dose_min": float(meta_aligned["dose_value"].min()),
        "dose_max": float(meta_aligned["dose_value"].max()),
        "groups_with_variation": int((var_by_group > 0).sum()),
    }
    return meta_aligned, summary

# Run gatekeeper (expecting 258 signatures based on previous step)
metadata_aligned, gate_summary = minimal_gatekeeper(
    exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=258
)

print("Gatekeeper summary:", gate_summary)


### Integrity Gatekeeper Conclusion

- 258 signatures aligned with the expression matrix  
- 12,328 genes retained  
- Dose column detected: `pert_dose` (range ≈ 0.01 – 10 µM)  
- 35 compound–cell groups show within-group dose variation  

The dataset passes all minimal integrity checks and is suitable for dose–response analyses.

---

### Dose–Response Setup

We now prepare the variables needed to model dose–response effects.  
For each signature, we create:  
- A **numeric dose value**  
- A **log10-transformed dose** for modeling trends  
- A simple **high vs. low bin** (median split within each compound–cell group)  

These variables provide the foundation for testing monotonic effects and for constructing contrasts between dose levels.


In [ ]:
# Check the global distribution of numeric doses
unique_doses = np.sort(metadata_aligned["dose_value"].unique())
median_dose = metadata_aligned["dose_value"].median()

print("Unique doses available:", unique_doses)
print("Global median dose:", median_dose)

In [ ]:
# Add numeric and log-transformed dose values
metadata_aligned["dose_value"] = pd.to_numeric(metadata_aligned["pert_dose"], errors="coerce")
metadata_aligned["log_dose"] = np.log10(metadata_aligned["dose_value"].clip(lower=1e-6))

# Create high/low bins by median split within each (compound × cell) group
metadata_aligned["dose_bin"] = np.where(
    metadata_aligned["dose_value"] >= 1, "high", "low"
)

# Merge compound names for readability
metadata_aligned = metadata_aligned.merge(
    compounds[["pert_id", "cmap_name"]],
    on="pert_id",
    how="left"
)

# Quick preview with compound names
metadata_aligned[["cmap_name", "pert_id", "cell_id", "dose_value", "log_dose", "dose_bin"]].head()


### Dose–Response Setup — Conclusion

- Dose variables created successfully: `dose_value`, `log_dose` (base-10), and `dose_bin` (median split within compound × cell).  
- Compound names (`cmap_name`) merged for readability (e.g., calcipotriol, calcitriol, ercalcitriol, tacalcitol).  
- Available doses in this subset span the expected log scale (0.1, 1.0, 10.0 µM).  
- The global median dose is **1.0 µM**, meaning that the median split used here effectively corresponds to a fixed threshold at 1 µM.  

These variables are ready to support both monotonic dose–response tests and binary high–low comparisons.
 
---

### Eligible Contrasts

We next identify compound–cell combinations that provide valid **high vs. low dose contrasts**.  
A contrast is considered eligible if both bins contain at least one signature, ensuring that statistical comparisons are possible.  
This step defines the set of comparisons that will drive the downstream differential expression and enrichment analyses.


In [ ]:
# Build the table of eligible high vs. low contrasts (compound × cell)
# A group is eligible if both bins have at least one signature.

# Count signatures per bin within each (compound, cell)
counts = (
    metadata_aligned
    .groupby(["pert_id", "cell_id", "dose_bin"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"high": "n_high", "low": "n_low"})
    .reset_index()
)

# Keep only groups with both high and low signatures
eligible = counts[(counts.get("n_high", 0) > 0) & (counts.get("n_low", 0) > 0)].copy()

# Attach compound names for readability
eligible = eligible.merge(
    compounds[["pert_id", "cmap_name"]],
    on="pert_id", how="left"
)

# Reorder and sort
eligible = eligible[["cmap_name", "cell_id", "n_high", "n_low", "pert_id"]]
eligible = eligible.sort_values(by=["n_high", "n_low", "cmap_name", "cell_id"], ascending=[False, False, True, True])

# Preview
eligible.head(15)

### Eligible Contrasts — Conclusion

- A robust set of **compound × cell** contrasts with both *high* and *low* dose bins was identified.  
- The best‑powered pairs are **calcitriol** in **HA1E (8/10)**, **MCF7 (8/8)** and **PC3 (5/6)**, followed by **maxacalcitol** (HA1E/MCF7 ≈ 5/6).  
- Additional contrasts include **ercalcitriol**, **tacalcitol**, **seocalcitol**, and **calcipotriol** across A549/MCF7/PC3/HA1E.

These contrasts are suitable for downstream **high vs. low** differential testing and for generating **ranked lists** used in enrichment.

---

## PCA of Vitamin D Signatures by Dose

To complement the previous PCA analyses stratified by compound and cell line, we next explore whether **dose level** (low vs. high) introduces systematic differences in transcriptional responses.  
The dataset was recently annotated with a `dose_bin` variable, defined using the median dose value (1 µM) as the cutoff. This allows us to group signatures into *low-dose* and *high-dose* categories across all compounds and cell lines.

By visualizing the PCA projection with points colored by dose, we aim to assess whether transcriptional variation is primarily driven by **treatment intensity**, or if dose has only a minor contribution compared to other factors such as cell identity.

In [ ]:
# Keep signatures with a valid dose bin and align rows/columns explicitly
sig_col = next(c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata_aligned.columns)
meta_pca = metadata_aligned.loc[metadata_aligned["dose_bin"].isin(["low", "high"]), [sig_col, "dose_bin"]].copy()
cols = meta_pca[sig_col].astype(str).tolist()

# Signatures × genes matrix for PCA (columns in the exact same order as metadata)
X = exp_matrix.loc[:, cols].T  # shape: (n_signatures, n_genes)

# Fit PCA on centered data (Level 5 are z-scores; no extra scaling required)
pca = PCA(n_components=2, random_state=0)
PC = pca.fit_transform(X)
ve = pca.explained_variance_ratio_ * 100

# Build plotting frame with guaranteed alignment
df_pca = pd.DataFrame(PC, columns=["PC1", "PC2"])
df_pca["dose_bin"] = meta_pca.set_index(sig_col).loc[cols, "dose_bin"].values

# Two well‑separated colors (not a rainbow)
palette = {"low": "#1f77b4",  # blue
           "high": "#d62728"} # red

plt.figure(figsize=(5, 3.5))
sns.scatterplot(
    data=df_pca,
    x="PC1", y="PC2",
    hue="dose_bin",
    palette=palette,
    s=35, alpha=0.85, edgecolor="none"
)
plt.xlabel(f"PC1 ({ve[0]:.1f}%)")
plt.ylabel(f"PC2 ({ve[1]:.1f}%)")
plt.title("PCA of Vitamin D signatures colored by dose")
plt.legend(title="Dose bin", frameon=False)
plt.tight_layout()
plt.show()

#### Interpretation of PCA by Dose

The PCA projection of Vitamin D signatures, colored by dose, reveals:

- **No strict global separation** between low- and high-dose treatments, indicating overlapping transcriptomic responses.  
- In the **PC3 cell line**, high-dose signatures (red) tend to be more dispersed and shifted away from the cluster center, suggesting **stronger transcriptional perturbations** at higher doses.  
- In other cell lines, the effect of dose is subtler, though some high-dose points still deviate slightly more from the main cloud.  

> Overall, the data suggest that higher doses amplify transcriptional changes, most evidently in PC3, while other cell lines show a less pronounced but consistent trend.

---

## PERMANOVA Analysis: Quantifying the Effects of Cell Line, Dose, and Compound

We applied PERMANOVA (Permutational Multivariate Analysis of Variance) on the Euclidean distance matrix of the expression profiles to quantify how much of the variance is explained by three main factors:  
- **Cell line** (tissue context),  
- **Dose** (low vs. high, based on a 1 µM threshold),  
- **Compound identity** (different Vitamin D analogs).  

This test reports:  
- **Pseudo-F statistic**: relative strength of the effect,  
- **p-value**: significance based on permutations,  
- **R²**: proportion of variance explained by each factor.

In [ ]:
# Scale expression matrix (signatures × genes)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(exp_matrix.T)

# Compute Euclidean distance matrix
dist_matrix = squareform(pdist(X_scaled, metric="euclidean"))
dm = DistanceMatrix(dist_matrix, ids=exp_matrix.columns.tolist())

# Metadata already aligned and includes dose_bin + compound
meta = metadata_aligned.set_index("sig_id")[["cell_id", "dose_bin", "cmap_name"]]

# Run PERMANOVA
res_cell = permanova(dm, meta, column="cell_id", permutations=999)
res_dose = permanova(dm, meta, column="dose_bin", permutations=999)
res_comp = permanova(dm, meta, column="cmap_name", permutations=999)

print("PERMANOVA results:")
print("\nCell line effect:\n", res_cell)
print("\nDose effect:\n", res_dose)
print("\nCompound effect:\n", res_comp)


### PERMANOVA Results: Cell Line, Dose, and Compound

The PERMANOVA analysis shows that:

- **Cell line** is the dominant source of variation (pseudo-F = 5.18, p = 0.001).  
- **Dose** has a weaker but significant effect (pseudo-F = 1.72, p = 0.016).  
- **Compound identity** also contributes modestly (pseudo-F = 1.30, p = 0.005).  

> Overall, transcriptomic variability is primarily driven by the cellular context, while both dose and compound exert secondary influences. The slightly stronger effect of dose supports treating Vitamin D analogs collectively as a single perturbation class.

---

## Two-way PERMANOVA: Joint Effects of Cell Line, Dose, and Compound

To disentangle the relative contributions of the three factors simultaneously, we performed a multi-factor PERMANOVA including:  
- **Cell line** (tissue context),  
- **Dose** (low vs. high, threshold at 1 µM),  
- **Compound identity** (Vitamin D analog).  

This approach estimates the variance explained by each factor **while adjusting for the others**, and also allows testing for **interactions** (e.g., whether the effect of dose depends on the cell line).

In [ ]:
# PERMANOVA within each cell line (dose and compound)
meta_all = metadata_aligned.set_index("sig_id")[["cell_id", "dose_bin", "cmap_name"]]

results = []
for cell in sorted(meta_all["cell_id"].unique()):
    cols = meta_all.index[meta_all["cell_id"] == cell]
    if len(cols) < 4:
        continue  # too few signatures to build a distance matrix

    # subset expression and metadata
    X = exp_matrix.loc[:, cols].T  # signatures × genes
    m = meta_all.loc[cols]

    # skip invalid groupings
    if m["dose_bin"].nunique() >= 2:
        Xs = StandardScaler().fit_transform(X)
        dm = DistanceMatrix(squareform(pdist(Xs, metric="euclidean")), ids=cols.tolist())
        res_dose_cell = permanova(dm, m[["dose_bin"]], column="dose_bin", permutations=999)
        results.append({"cell_id": cell, "factor": "dose_bin",
                        "groups": m["dose_bin"].nunique(),
                        "pseudoF": float(res_dose_cell["test statistic"]),
                        "pvalue": float(res_dose_cell["p-value"])})
    if m["cmap_name"].nunique() >= 2:
        Xs = StandardScaler().fit_transform(X)
        dm = DistanceMatrix(squareform(pdist(Xs, metric="euclidean")), ids=cols.tolist())
        res_comp_cell = permanova(dm, m[["cmap_name"]], column="cmap_name", permutations=999)
        results.append({"cell_id": cell, "factor": "cmap_name",
                        "groups": m["cmap_name"].nunique(),
                        "pseudoF": float(res_comp_cell["test statistic"]),
                        "pvalue": float(res_comp_cell["p-value"])})

# Pretty print
if results:
    df_res = pd.DataFrame(results).sort_values(["factor", "cell_id"])
    print(df_res.to_string(index=False))
else:
    print("No valid within-cell tests (check group counts).")


#### PERMANOVA Results Within Each Cell Line

When testing within each cell line, the following patterns emerge:

- **Compound effect**: significant in A549, PC3, and U2OS, but not in HA1E or MCF7.  
- **Dose effect**: significant only in PC3 and marginally in MCF7; not significant in A549, HA1E, or U2OS.  

> Overall, compound identity shows detectable variability in several cell types, whereas the dose effect is more restricted, being robust mainly in PC3. These results reinforce the dominant role of the cellular context and highlight PC3 as particularly sensitive to dose-dependent transcriptomic changes.

---

In [ ]:
# Step 1 — per‑cell mean profile (REPLACE this whole cell)

# Identify the signature id column
sig_col = next(c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata_aligned.columns)

# Mean z‑score per cell line (genes × cells)
expr_by_cell = exp_matrix.groupby(metadata_aligned.set_index(sig_col)["cell_id"], axis=1).mean()

# Build result table and attach IDs/symbols robustly
res_cell = expr_by_cell.copy()
# ensure gene_id column (string) exists for downstream merges
res_cell.insert(0, "gene_id", res_cell.index.astype(str))

# try to map gene_symbol if available, coercing both sides to string
sym_col = next((c for c in ["gene_symbol", "pr_gene_symbol", "symbol"] if c in gene_info.columns), None)
if "gene_id" in gene_info.columns and sym_col is not None:
    gi_map = (
        gene_info.assign(gene_id=gene_info["gene_id"].astype(str))
                 .drop_duplicates(subset=["gene_id"])
                 .set_index("gene_id")[sym_col]
                 .to_dict()
    )
    res_cell.insert(1, "gene_symbol", res_cell["gene_id"].map(gi_map))
else:
    # keep column for downstream code even if symbols are unavailable
    res_cell.insert(1, "gene_symbol", np.nan)


In [ ]:
# Step 2 — view top up/down per cell (replace the previous cell with this one)

def top_genes_cell(cell_id, n=10):
    """
    Build a small table for one cell line including:
    - gene_symbol (if available in res_cell)
    - gene_id (from the index)
    - mean_z (average moderated z-score across treated signatures)
    """
    # Bring along gene_symbol if it exists
    cols = [cell_id]
    if "gene_symbol" in res_cell.columns:
        cols = ["gene_symbol"] + cols

    df = res_cell[cols].copy()
    df = df.rename(columns={cell_id: "mean_z"})
    df["gene_id"] = res_cell.index.astype(str)

    df = df.sort_values("mean_z", ascending=False)
    top_up = df.head(n)
    top_dn = df.tail(n)
    return top_up, top_dn

# Example: PC3 and MCF7
for c in ["PC3", "MCF7"]:
    if c in res_cell.columns:
        up, dn = top_genes_cell(c, n=10)
        cols_to_show = [col for col in ["gene_symbol", "gene_id", "mean_z"] if col in up.columns]
        print(f"\nTop UP in {c}")
        print(up[cols_to_show])
        print(f"\nTop DOWN in {c}")
        print(dn[cols_to_show])


### Build ranked lists per cell + consensus “core VDR” by vote-count

For each cell, rank genes by mean z‑score (treatment vs control already encoded in L1000).
Then, take the top‑N up/down per cell and count how often each gene appears across cells.


In [ ]:
# Ranked lists per cell (dict: cell -> Series mean_z sorted desc)
rank_by_cell = {
    cell: res_cell[cell].sort_values(ascending=False)
    for cell in res_cell.columns
    if cell not in ["gene_symbol", "gene_id"]
}

def top_sets(cell, n=50):
    s = rank_by_cell[cell]
    up_ids = set(s.head(n).index.astype(str))
    dn_ids = set(s.tail(n).index.astype(str))
    return up_ids, dn_ids

# Vote-count across all cells
from collections import Counter

N_TOP = 50  # adjust if you want stricter/looser consensus
up_votes = Counter()
dn_votes = Counter()

for cell in rank_by_cell.keys():
    up_ids, dn_ids = top_sets(cell, n=N_TOP)
    up_votes.update(up_ids)
    dn_votes.update(dn_ids)

# Build consensus tables (≥2 cells by default)
VOTE_MIN = 2
# create a symbol map with string keys (fix dtype mismatch)
sym_map = None
if "gene_symbol" in res_cell.columns:
    _s = res_cell["gene_symbol"].copy()
    _s.index = _s.index.astype(str)  # <-- critical fix
    sym_map = _s.to_dict()


def build_consensus_table(counter, kind="up"):
    # Keep only genes with at least VOTE_MIN votes
    items = [(str(gid), cnt) for gid, cnt in counter.items() if cnt >= VOTE_MIN]
    df = pd.DataFrame(items, columns=["gene_id", f"votes_{kind}"])

    # attach symbol if available (keys are strings)
    if sym_map is not None:
        df["gene_symbol"] = df["gene_id"].map(sym_map)
    
    # average effect across cells (sign-aware summary)
    cols = [c for c in res_cell.columns if c not in ["gene_symbol", "gene_id"]]
    avg_effect = (
        res_cell[cols]
        .set_index(res_cell.index.astype(str))  # ensure string index
        .mean(axis=1)
        .rename("global_mean_z")
        .reset_index()
        .rename(columns={"index": "gene_id"})
    )
    
    df = df.merge(avg_effect, on="gene_id", how="left")
    
    # order by votes then effect
    df = df.sort_values([f"votes_{kind}", "global_mean_z"], ascending=[False, kind=="down"]).reset_index(drop=True)
    return df


consensus_up   = build_consensus_table(up_votes, kind="up")
consensus_down = build_consensus_table(dn_votes, kind="down")

print("Consensus UP (appears in ≥2 cells among top-N):")
print(consensus_up.head(20)[[c for c in ["gene_symbol","gene_id","votes_up","global_mean_z"] if c in consensus_up.columns]])

print("Consensus DOWN (appears in ≥2 cells among bottom-N):")
print(consensus_down.head(20)[[c for c in ["gene_symbol","gene_id","votes_down","global_mean_z"] if c in consensus_down.columns]])